In [7]:
import pandas as pd

pd.set_option("display.max_columns", None)

# F1 CSVs use "\N" for missing values
na = ["\\N"]
results      = pd.read_csv("f1/results.csv", na_values=na)
races        = pd.read_csv("f1/races.csv", na_values=na)
drivers      = pd.read_csv("f1/drivers.csv", na_values=na)
constructors = pd.read_csv("f1/constructors.csv", na_values=na)
circuits     = pd.read_csv("f1/circuits.csv", na_values=na)
status       = pd.read_csv("f1/status.csv", na_values=na)

for name, df in {"results": results, "races": races, "drivers": drivers,
                 "constructors": constructors, "circuits": circuits}.items():
    print(f"\n=== {name} {df.shape} ===")
    print(df.head(3))
    print(df.isna().sum()[df.isna().sum() > 0])


=== results (26759, 18) ===
   resultId  raceId  driverId  constructorId  number  grid  position  \
0         1      18         1              1    22.0     1       1.0   
1         2      18         2              2     3.0     5       2.0   
2         3      18         3              3     7.0     7       3.0   

  positionText  positionOrder  points  laps         time  milliseconds  \
0            1              1    10.0    58  1:34:50.616     5690616.0   
1            2              2     8.0    58       +5.478     5696094.0   
2            3              3     6.0    58       +8.163     5698779.0   

   fastestLap  rank fastestLapTime  fastestLapSpeed  statusId  
0        39.0   2.0       1:27.452          218.300         1  
1        41.0   3.0       1:27.739          217.586         1  
2        41.0   5.0       1:28.090          216.719         1  
number                 6
position           10953
time               19079
milliseconds       19079
fastestLap         18507
rank

In [11]:
df = (
    results
    .merge(races[["raceId", "year", "round", "circuitId", "name", "date"]]
           .rename(columns={"name": "race_name"}),
           on="raceId", how="left")
    .merge(drivers[["driverId", "forename", "surname", "nationality", "dob"]],
           on="driverId", how="left")
    .merge(constructors[["constructorId", "name"]]
           .rename(columns={"name": "constructor"}),
           on="constructorId", how="left")
    .merge(circuits[["circuitId", "name", "country"]]
           .rename(columns={"name": "circuit", "country": "circuit_country"}),
           on="circuitId", how="left")
)

df["driver"] = df["forename"] + " " + df["surname"]
df["date"] = pd.to_datetime(df["date"])
df["podium"] = (df["positionOrder"] <= 3).astype(int)

print(df.shape)
df[["year", "race_name", "driver", "constructor", "grid", "positionOrder", "podium"]].head()

(26759, 32)


,year,race_name,driver,constructor,grid,positionOrder,podium
0,2008,Australian Grand Prix,Lewis Hamilton,McLaren,1,1,1
1,2008,Australian Grand Prix,Nick Heidfeld,BMW Sauber,5,2,1
2,2008,Australian Grand Prix,Nico Rosberg,Williams,7,3,1
3,2008,Australian Grand Prix,Fernando Alonso,Renault,11,4,0
4,2008,Australian Grand Prix,Heikki Kovalainen,McLaren,3,5,0


In [13]:
# 1. Merge didn't duplicate rows (should print True)
print(df.shape[0] == results.shape[0])

# 2. Class balance of the target
print(df["podium"].value_counts())
print(df["podium"].value_counts(normalize=True).round(3))

# 3. Podium rate by starting grid slot
print(df[df["grid"].between(1, 10)].groupby("grid")["podium"].mean().round(3))

# 4. Coverage
print(df["year"].min(), "to", df["year"].max(), "|", df["raceId"].nunique(), "races")

True
podium
0    23362
1     3397
Name: count, dtype: int64
podium
0    0.873
1    0.127
Name: proportion, dtype: float64
grid
1     0.635
2     0.556
3     0.463
4     0.341
5     0.250
6     0.186
7     0.130
8     0.099
9     0.083
10    0.058
Name: podium, dtype: float64
1950 to 2024 | 1125 races


In [15]:
per_race = df.groupby("raceId")["podium"].sum()
extra = per_race[per_race > 3]
print(len(extra), "races with more than 3 podium rows")
print(df[df["raceId"].isin(extra.index)]
      .query("podium == 1")[["year", "race_name", "driver", "positionOrder"]]
      .sort_values(["year", "race_name"]).head(15))

18 races with more than 3 podium rows
       year           race_name                 driver  positionOrder
20149  1950  Italian Grand Prix            Nino Farina              1
20150  1950  Italian Grand Prix        Dorino Serafini              2
20151  1950  Italian Grand Prix          Luigi Fagioli              3
20182  1950  Italian Grand Prix         Alberto Ascari              2
19917  1951   French Grand Prix          Luigi Fagioli              1
19918  1951   French Grand Prix         Alberto Ascari              2
19919  1951   French Grand Prix        Luigi Villoresi              3
20185  1951   French Grand Prix            Juan Fangio              1
20186  1951   French Grand Prix  José Froilán González              2
19871  1951    Indianapolis 500            Lee Wallard              1
19872  1951    Indianapolis 500           Mike Nazaruk              2
19873  1951    Indianapolis 500           Jack McGrath              3
20184  1951    Indianapolis 500            Manny Ayu

In [17]:
print(df[df["raceId"].isin(extra.index)]["year"].value_counts().sort_index())

year
1950     29
1951     83
1953     67
1954     46
1955     96
1956    112
1957     22
1960     23
Name: count, dtype: int64


In [19]:
modern = df[df["year"] >= 2000].copy()

print(modern.shape)
print(modern["podium"].value_counts())
print(modern["podium"].value_counts(normalize=True).round(3))

# Should now be exactly 3 podium rows per race
print((modern.groupby("raceId")["podium"].sum() != 3).sum(), "races without exactly 3 podiums")

(10079, 32)
podium
0    8642
1    1437
Name: count, dtype: int64
podium
0    0.857
1    0.143
Name: proportion, dtype: float64
0 races without exactly 3 podiums


In [21]:
modern.to_csv("data/f1_modern.csv", index=False)